<a href="https://colab.research.google.com/github/etcex2969-spec/-AIFFEL_quest_eng/blob/main/%EB%8F%8CPIPLINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

                              RAG Pipeline — Dataset Benchmark & Evaluation

dataset_experiments.py
│
├── SECTION 1 — 검색(Retrieval) 품질 측정
│   ├── 1-A  MS MARCO         load_msmarco()           run_msmarco_experiment()
│   ├── 1-B  BEIR Benchmark   load_beir_subset()       run_beir_experiment()
│   └── 1-C  HotpotQA         load_hotpotqa()          run_hotpotqa_experiment()
│
├── SECTION 2 — 도메인 특화
│   ├── 2-A  FinQA             load_finqa()             run_finqa_experiment()
│   ├── 2-B  CodeSearchNet     load_codesearchnet()     run_codesearchnet_experiment()
│   └── 2-C  PubMedQA          load_pubmedqa()          run_pubmedqa_experiment()
│
└── SECTION 3 — 공통 평가 유틸
    ├── exact_match()          EM 계산
    ├── f1_score()             Token F1 계산
    ├── recall_at_k()          Recall@K 계산
    ├── save_results_csv()     결과 CSV 저장
    └── print_summary()        요약 출력

In [ ]:
pip install datasets openai

In [38]:
from dataset_experiments import load_finqa, run_finqa_experiment, print_summary, save_results_csv

# 실험 실행
results = run_finqa_experiment(pipeline, sample_size=100)

# 요약 출력
print_summary(results, "Dummy Data RAG Experiment")

# CSV 저장
save_results_csv(results, "dummy_rag_result.csv")

{"component": "RAGPipeline", "question_preview": "Python \ub515\uc154\ub108\ub9ac\uc5d0\uc11c \ud0a4\ub97c \ucd94\uac00\ud558\ub294 \ubc29\ubc95\uc740?", "event": "pipeline_start", "request_id": "9dbe2197-4fb2-43a8-8a37-e97ef3494c9c", "level": "info", "timestamp": "2026-06-04T08:04:32.090324Z"}
{"component": "RAGPipeline", "status": "agent_ready:retrieval_tool", "event": "step_1_init_agent", "request_id": "9dbe2197-4fb2-43a8-8a37-e97ef3494c9c", "level": "info", "timestamp": "2026-06-04T08:04:32.092266Z"}
{"component": "RAGPipeline", "tool": "retrieval_tool", "event": "step_2_decide_tool", "request_id": "9dbe2197-4fb2-43a8-8a37-e97ef3494c9c", "level": "info", "timestamp": "2026-06-04T08:04:32.093648Z"}
{"component": "RAGPipeline", "event": "step_3_run_rag", "request_id": "9dbe2197-4fb2-43a8-8a37-e97ef3494c9c", "level": "info", "timestamp": "2026-06-04T08:04:32.094302Z"}
{"component": "RAGClient", "url": "https://rag-tool.example.com/api/rag", "question": "Python \ub515\uc154\ub108\ub9ac

In [24]:
import pandas as pd

df_results = pd.DataFrame(results)
display(df_results.head())

,id,question,gold_answer,predicted_answer
0,0,Python 딕셔너리에서 키를 추가하는 방법은?,`dict[key] = value` 구문을 사용합니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
1,1,Git에서 변경사항을 스테이징하는 명령어는?,`git add <파일이름>` 입니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
2,2,AWS EC2 인스턴스란 무엇인가요?,클라우드에서 안전하고 크기 조정 가능한 컴퓨팅 파워를 제공하는 웹 서비스입니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
3,3,Kubernetes에서 파드(Pod)를 배포하는 방법은?,`kubectl apply -f <yaml파일>` 명령어를 사용합니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
4,4,SQL에서 모든 데이터를 선택하는 쿼리는?,`SELECT * FROM <테이블이름>;` 쿼리를 사용합니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)


In [31]:
from dataset_experiments import exact_match, f1_score

# Calculate Exact Match and F1 Score for each row
df_results['em_score'] = df_results.apply(lambda row: exact_match(row['predicted_answer'], row['gold_answer']), axis=1)
df_results['f1_score'] = df_results.apply(lambda row: f1_score(row['predicted_answer'], row['gold_answer']), axis=1)

# Calculate and display overall average scores
avg_em = df_results['em_score'].mean()
avg_f1 = df_results['f1_score'].mean()

print(f"\n--- 분석 요약 ---")
print(f"평균 Exact Match (EM): {avg_em:.4f}")
print(f"평균 F1 스코어: {avg_f1:.4f}")
print("------------------")

# Display some examples where predicted_answer does not match gold_answer
mismatched_results = df_results[df_results['em_score'] == 0.0]

if not mismatched_results.empty:
    print("\n--- 예측과 정답이 일치하지 않는 예시 ---")
    for i, row in mismatched_results.head(5).iterrows(): # Display up to 5 mismatched examples
        print(f"Q: {row['question']}")
        print(f"정답: {row['gold_answer']}")
        print(f"예측: {row['predicted_answer']}")
        print(f"EM: {row['em_score']:.2f}, F1: {row['f1_score']:.2f}")
        print("---")
else:
    print("\n모든 예측이 정답과 일치합니다.")

# Display the updated DataFrame with scores
print("\n--- 결과 DataFrame (상위 5개) ---")
display(df_results.head())


--- 분석 요약 ---
평균 Exact Match (EM): 0.0000
평균 F1 스코어: 0.0000
------------------

--- 예측과 정답이 일치하지 않는 예시 ---
Q: Python 딕셔너리에서 키를 추가하는 방법은?
정답: `dict[key] = value` 구문을 사용합니다.
예측: [시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
EM: 0.00, F1: 0.00
---
Q: Git에서 변경사항을 스테이징하는 명령어는?
정답: `git add <파일이름>` 입니다.
예측: [시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
EM: 0.00, F1: 0.00
---
Q: AWS EC2 인스턴스란 무엇인가요?
정답: 클라우드에서 안전하고 크기 조정 가능한 컴퓨팅 파워를 제공하는 웹 서비스입니다.
예측: [시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
EM: 0.00, F1: 0.00
---
Q: Kubernetes에서 파드(Pod)를 배포하는 방법은?
정답: `kubectl apply -f <yaml파일>` 명령어를 사용합니다.
예측: [시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
EM: 0.00, F1: 0.00
---
Q: SQL에서 모든 데이터를 선택하는 쿼리는?
정답: `SELECT * FROM <테이블이름>;` 쿼리를 사용합니다.
예측: [시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)
EM: 0.00, F1: 0.00
---

--- 결과 DataFrame (상위 5개) ---


,id,question,gold_answer,predicted_answer,em_score,f1_score
0,0,Python 딕셔너리에서 키를 추가하는 방법은?,`dict[key] = value` 구문을 사용합니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1),0.0,0.0
1,1,Git에서 변경사항을 스테이징하는 명령어는?,`git add <파일이름>` 입니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1),0.0,0.0
2,2,AWS EC2 인스턴스란 무엇인가요?,클라우드에서 안전하고 크기 조정 가능한 컴퓨팅 파워를 제공하는 웹 서비스입니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1),0.0,0.0
3,3,Kubernetes에서 파드(Pod)를 배포하는 방법은?,`kubectl apply -f <yaml파일>` 명령어를 사용합니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1),0.0,0.0
4,4,SQL에서 모든 데이터를 선택하는 쿼리는?,`SELECT * FROM <테이블이름>;` 쿼리를 사용합니다.,[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1),0.0,0.0


In [25]:
%%writefile dataset_experiments.py
import pandas as pd
# from datasets import load_dataset # No longer needed

def load_finqa(sample_size: int = 100):
    """
    Dummy data loader for RAG experiment.
    (question, context, answer_text)
    """
    dummy_data = [
        {"question": "Python 딕셔너리에서 키를 추가하는 방법은?", "context": "Python 딕셔너리는 키-값 쌍을 저장하는 컬렉션입니다. 딕셔너리에 새 키-값 쌍을 추가하려면 `dict[key] = value` 구문을 사용합니다.", "answer_text": "`dict[key] = value` 구문을 사용합니다."},
        {"question": "Git에서 변경사항을 스테이징하는 명령어는?", "context": "Git은 분산 버전 관리 시스템입니다. `git add <파일이름>` 명령어를 사용하여 변경사항을 스테이징 영역에 추가할 수 있습니다.", "answer_text": "`git add <파일이름>` 입니다."},
        {"question": "AWS EC2 인스턴스란 무엇인가요?", "context": "Amazon Elastic Compute Cloud (Amazon EC2)는 안전하고 크기 조정이 가능한 컴퓨팅 파워를 클라우드에서 제공하는 웹 서비스입니다.", "answer_text": "클라우드에서 안전하고 크기 조정 가능한 컴퓨팅 파워를 제공하는 웹 서비스입니다."},
        {"question": "Kubernetes에서 파드(Pod)를 배포하는 방법은?", "context": "Kubernetes는 컨테이너화된 워크로드를 자동 배포, 스케일링 및 관리하는 오픈소스 시스템입니다. 파드는 쿠버네티스에서 생성 및 배포할 수 있는 가장 작은 배포 단위입니다. `kubectl apply -f <yaml파일>` 명령어를 사용합니다.", "answer_text": "`kubectl apply -f <yaml파일>` 명령어를 사용합니다."},
        {"question": "SQL에서 모든 데이터를 선택하는 쿼리는?", "context": "SQL은 관계형 데이터베이스 관리 시스템에서 데이터를 관리하기 위한 표준 언어입니다. 테이블의 모든 열과 모든 행을 가져오려면 `SELECT * FROM <테이블이름>;` 쿼리를 사용합니다.", "answer_text": "`SELECT * FROM <테이블이름>;` 쿼리를 사용합니다."},
    ]
    # Replicate dummy data to reach sample_size if needed
    if sample_size > len(dummy_data):
        return (dummy_data * (sample_size // len(dummy_data) + 1))[:sample_size]
    return dummy_data[:sample_size]


def run_finqa_experiment(pipeline, sample_size: int = 100) -> list[dict]:
    """
    Dummy 데이터셋으로 RAG 실험을 수행합니다.
    """
    rows = load_finqa(sample_size)
    results = []
    for i, row in enumerate(rows):
        question = row["question"]
        gold_answer = row["answer_text"]

        # RAG Pipeline 실행
        # RAG Pipeline은 'context' 인자를 받지 않고 내부적으로 retrieve를 수행함
        # 따라서, 여기서는 'question'만 전달하고, RAG_CONTEXT_SNIPPET은 pipeline 설정에서
        # 관리되거나, retrieval_tool이 알아서 찾아오도록 가정합니다.
        predicted_answer = pipeline.run(question)

        results.append({
            "id": i,
            "question": question,
            "gold_answer": gold_answer,
            "predicted_answer": predicted_answer,
        })
    return results

def exact_match(predicted: str, gold: str) -> float:
    return float(predicted.strip().lower() == gold.strip().lower())

def f1_score(predicted: str, gold: str) -> float:
    # 간단한 토큰 F1 스코어 구현 (실제 F1은 더 복잡함)
    pred_tokens = set(predicted.lower().split())
    gold_tokens = set(gold.lower().split())
    common = len(pred_tokens.intersection(gold_tokens))
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    if precision + recall == 0:
        return 0.0
    return (2 * precision * recall) / (precision + recall)


def print_summary(results: list[dict], experiment_name: str) -> None:
    """실험 결과를 요약하여 출력합니다."""
    if not results:
        print(f"[{experiment_name}] No results to summarize.")
        return

    em_scores = [exact_match(r["predicted_answer"], r["gold_answer"]) for r in results]
    f1_scores = [f1_score(r["predicted_answer"], r["gold_answer"]) for r in results]

    avg_em = sum(em_scores) / len(em_scores)
    avg_f1 = sum(f1_scores) / len(f1_scores)

    print(f"\n--- {experiment_name} 실험 요약 ---")
    print(f"샘플 수: {len(results)}")
    print(f"평균 Exact Match (EM): {avg_em:.4f}")
    print(f"평균 F1 스코어: {avg_f1:.4f}")
    print("-" * 30)
    for i, res in enumerate(results[:5]): # 상위 5개만 출력
        print(f"Q: {res['question']}")
        print(f"정답: {res['gold_answer']}")
        print(f"예측: {res['predicted_answer']}")
        print(f"EM: {em_scores[i]:.2f}, F1: {f1_scores[i]:.2f}")
        print("---")

def save_results_csv(results: list[dict], filename: str) -> None:
    """실험 결과를 CSV 파일로 저장합니다."""
    df = pd.DataFrame(results)
    df.to_csv(filename, index=False)
    print(f"결과가 '{filename}'에 저장되었습니다.")

Overwriting dataset_experiments.py


In [39]:
# 데이터셋 직접 로딩은 현재 문제가 있어 RAG 실험에서는 가상 데이터를 사용합니다.
# from datasets import load_dataset

# ds = load_dataset("klue/squad_kor_v1", split="validation[:200]")

print("Dataset 'klue/squad_kor_v1' could not be loaded directly for testing.")
print("The RAG experiment in the next cell will use dummy data defined in `dataset_experiments.py`.")
# for item in ds:
#     question = item["question"]
#     context  = item["context"]
#     answer   = item["answers"]["text"]
#
#     result = pipeline.run(question)
#     print(f"Q: {question}")
#     print(f"정답: {answer[0]}")
#     print(f"모델: {result}")
#     print("---")

Dataset 'klue/squad_kor_v1' could not be loaded directly for testing.
The RAG experiment in the next cell will use dummy data defined in `dataset_experiments.py`.


In [40]:
import importlib
import dataset_experiments
importlib.reload(dataset_experiments)
print("dataset_experiments module reloaded successfully.")

dataset_experiments module reloaded successfully.


In [12]:
!pip install -q structlog pydantic tenacity requests openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00


In [37]:
%%writefile rag_pipeline.py
from __future__ import annotations

import uuid
import logging
from typing import Optional, Protocol, runtime_checkable

import requests
import structlog
from pydantic import BaseModel, Field, field_validator
from tenacity (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

# ---------------------------------------------------------------------------
# 1. 설정 (Pydantic) — 환경 변수에서 자동 로드
# ---------------------------------------------------------------------------

class RAGConfig(BaseModel):
    """모든 설정을 한 곳에서 관리. 환경 변수 또는 직접 주입 가능."""

    rag_api_base: str = Field(
        default="https://rag-tool.example.com/api",
        validation_alias="RAG_API_BASE",
    )
    rag_api_key: str = Field(default="", validation_alias="RAG_API_KEY")
    rag_tool_name: str = Field(
        default="retrieval_tool", validation_alias="RAG_TOOL_NAME"
    )
    rag_context_snippet: str = Field(
        default="[RAG 지식] 기본 컨텍스트입니다.",
        validation_alias="RAG_CONTEXT_SNIPPET",
    )
    openai_api_key: str = Field(default="", validation_alias="OPENAI_API_KEY")
    openai_model: str = Field(default="gpt-4o-mini", validation_alias="OPENAI_MODEL")

    # 재시도 설정
    retry_attempts: int = Field(default=3, validation_alias="RAG_RETRY_COUNT")
    retry_min_wait: float = Field(default=1.0, validation_alias="RAG_BACKOFF_SEC")
    retry_max_wait: float = Field(default=10.0, validation_alias="RAG_MAX_BACKOFF_SEC")
    timeout_sec: float = Field(default=5.0, validation_alias="RAG_TIMEOUT_SEC")

    model_config = {"populate_by_name": True}

    # ── [추가] 유효성 검증 ──────────────────────────────────────
    @field_validator("retry_attempts")
    @classmethod
    def _validate_retry(cls, v: int) -> int:
        if v < 0:
            raise ValueError(f"retry_attempts 는 0 이상이어야 합니다. (입력값: {v})")
        return v

    @field_validator("timeout_sec")
    @classmethod
    def _validate_timeout(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"timeout_sec 는 0보다 커야 합니다. (입력값: {v})")
        return v

    @field_validator("rag_api_base")
    @classmethod
    def _validate_api_base(cls, v: str) -> str:
        if not v or not v.strip():
            raise ValueError("rag_api_base 는 빈 문자열일 수 없습니다.")
        return v

    @classmethod
    def from_env(cls) -> "RAGConfig":
        """환경 변수에서 설정을 로드합니다."""
        import os
        return cls(
            RAG_API_BASE=os.getenv("RAG_API_BASE", "https://rag-tool.example.com/api"),
            RAG_API_KEY=os.getenv("RAG_API_KEY", ""),
            RAG_TOOL_NAME=os.getenv("RAG_TOOL_NAME", "retrieval_tool"),
            RAG_CONTEXT_SNIPPET=os.getenv(
                "RAG_CONTEXT_SNIPPET", "[RAG 지식] 기본 컨텍스트입니다."
            ),
            OPENAI_API_KEY=os.getenv("OPENAI_API_KEY", ""),
            OPENAI_MODEL=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            RAG_RETRY_COUNT=int(os.getenv("RAG_RETRY_COUNT", "3")),
            RAG_BACKOFF_SEC=float(os.getenv("RAG_BACKOFF_SEC", "1.0")),
            RAG_MAX_BACKOFF_SEC=float(os.getenv("RAG_MAX_BACKOFF_SEC", "10.0")),
            RAG_TIMEOUT_SEC=float(os.getenv("RAG_TIMEOUT_SEC", "5.0")),
        )


# ---------------------------------------------------------------------------
# 2. 구조적 로깅 (structlog) — JSON 포맷, request_id 컨텍스트 바인딩
# ---------------------------------------------------------------------------

def _configure_logging() -> None:
    """
    [수정] 모듈 임포트 시 전역 로그 설정이 즉시 적용되는 부작용을 방지.
    테스트 환경에서는 호출하지 않아도 되며, 운영 엔트리포인트에서만 호출한다.
    """
    structlog.configure(
        processors=[
            structlog.contextvars.merge_contextvars,
            structlog.processors.add_log_level,
            structlog.processors.TimeStamper(fmt="iso"),
            structlog.processors.JSONRenderer(),
        ],
        wrapper_class=structlog.make_filtering_bound_logger(logging.INFO),
        context_class=dict,
        logger_factory=structlog.PrintLoggerFactory(),
    )

logger = structlog.get_logger()


# ---------------------------------------------------------------------------
# 3. 클라이언트 인터페이스 (Protocol) — 의존성 주입(DI) 기반
# ---------------------------------------------------------------------------

@runtime_checkable
class RAGClientProtocol(Protocol):
    def retrieve(self, context: str, question: str) -> str: ...


@runtime_checkable
class LLMClientProtocol(Protocol):
    def infer(self, prompt: str, temperature: float = 0.1) -> str: ...


# ---------------------------------------------------------------------------
# 4. RAG 클라이언트 — tenacity 재시도 + 세분화된 예외 처리
# ---------------------------------------------------------------------------

class RAGClient:
    """외부 RAG REST API 호출 클라이언트."""

    def __init__(self, config: RAGConfig) -> None:
        self._config = config
        self._log = logger.bind(component="RAGClient")
        # [수정] retry 데코레이터를 __init__ 에서 한 번만 생성 → 매 호출 재생성 방지
        self._retrying_call = retry(
            stop=stop_after_attempt(config.retry_attempts + 1),
            wait=wait_exponential(
                multiplier=config.retry_min_wait,
                max=config.retry_max_wait,
            ),
            retry=retry_if_exception_type(
                (requests.exceptions.Timeout, requests.exceptions.ConnectionError)
            ),
            before_sleep=before_sleep_log(
                logging.getLogger("tenacity"), logging.WARNING
            ),
            reraise=True,
        )(self._call_once)

    def retrieve(self, context: str, question: str) -> str:
        """tenacity 재시도 래퍼를 통해 RAG API 를 호출한다."""
        return self._retrying_call(context, question)

    def _call_once(self, context: str, question: str) -> str:
        """[수정] 실제 HTTP 호출 로직을 별도 메서드로 분리. 재시도 대상은 이 메서드."""
        cfg = self._config
        url = f"{cfg.rag_api_base}/rag"

        # 예시 URL일 경우 모의 응답 반환
        if "rag-tool.example.com" in cfg.rag_api_base:
            self._log.info("rag_api_call_mock", url=url, question=question)
            # 더미 컨텍스트 생성 (질문과 관련된 응답처럼 보이게)
            mock_context_map = {
                "Python 딕셔너리에서 키를 추가하는 방법은?": "Python 딕셔너리에서 키를 추가하려면 `dict[key] = value` 구문을 사용합니다. 이 방법으로 새로운 키와 값을 할당할 수 있습니다. 예를 들어, `my_dict = {}; my_dict['new_key'] = 'new_value'`와 같이 사용합니다.",
                "Git에서 변경사항을 스테이징하는 명령어는?": "Git에서 변경사항을 스테이징 영역에 추가하는 명령어는 `git add <파일이름>`입니다. 모든 변경사항을 스테이징하려면 `git add .`를 사용합니다.",
                "AWS EC2 인스턴스란 무엇인가요?": "AWS EC2 인스턴스는 아마존 웹 서비스(AWS) 클라우드에서 제공하는 가상 서버입니다. 확장 가능하고 안전하며 유연한 컴퓨팅 용량을 제공합니다.",
                "Kubernetes에서 파드(Pod)를 배포하는 방법은?": "Kubernetes에서 파드(Pod)를 배포하려면 YAML 파일로 파드 정의를 작성한 후 `kubectl apply -f <yaml파일 경로>` 명령어를 사용합니다. 파드는 쿠버네티스에서 실행되는 워크로드의 가장 작은 단위입니다.",
                "SQL에서 모든 데이터를 선택하는 쿼리는?": "SQL에서 테이블의 모든 열과 모든 행을 선택하는 쿼리는 `SELECT * FROM <테이블이름>;`입니다. 이 쿼리는 데이터 검토 시 유용합니다."
            }
            return mock_context_map.get(question, "[모의 응답] 요청하신 정보에 대한 모의 컨텍스트입니다.")

        headers = {
            "Authorization": f"Bearer {cfg.rag_api_key}" if cfg.rag_api_key else "",
            "Content-Type": "application/json",
        }
        payload = {
            "context": context,
            "question": question,
            "options": {"include_sources": True},
        }
        self._log.info("rag_api_call", url=url)
        resp = requests.post(
            url, json=payload, headers=headers, timeout=cfg.timeout_sec
        )
        # 401/403 → 즉시 중단 (재시도 불가)
        if resp.status_code in (401, 403):
            raise PermissionError(
                f"RAG API 인증 실패: {resp.status_code}. 즉시 중단."
            )
        resp.raise_for_status()
        data = resp.json()
        return data.get("context", context)



# ---------------------------------------------------------------------------
# 5. LLM 클라이언트 — OpenAI 호출 래퍼
# ---------------------------------------------------------------------------

class OpenAILLMClient:
    """OpenAI ChatCompletion 호출 클라이언트."""

    def __init__(self, config: RAGConfig) -> None:
        self._config = config
        self._log = logger.bind(component="OpenAILLMClient")

    def infer(self, prompt: str, temperature: float = 0.1) -> str:
        if not self._config.openai_api_key:
            self._log.warning("openai_key_missing", fallback="simulation_mode")
            return f"[시뮬레이션] 프롬프트 수신 완료 (temperature={temperature})"

        try:
            from openai import OpenAI  # type: ignore
            client = OpenAI(api_key=self._config.openai_api_key)
            self._log.info("llm_infer_start", model=self._config.openai_model)
            response = client.chat.completions.create(
                model=self._config.openai_model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=1024,
            )
            return response.choices[0].message.content.strip()
        except Exception as exc:
            self._log.error("llm_infer_failed", error=str(exc))
            raise


# ---------------------------------------------------------------------------
# 6. 프롬프트 빌더 — Persona + Constraint + Few-shot
# ---------------------------------------------------------------------------

FEW_SHOT_EXAMPLES = """
[예시 1]
질문: GitHub Actions에서 Python 버전을 지정하는 방법은?
답변: `actions/setup-python` 액션에서 `python-version: '3.11'`로 지정합니다.

[예시 2]
질문: 존재하지 않는 기능에 대한 질문입니다.
답변: 죄송합니다. 제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다.
"""

SYSTEM_PROMPT_TEMPLATE = """
당신은 숙련된 소프트웨어 엔지니어링 및 MLOps 전문가이자 기술 문서 작성자입니다. 제공된 정보만을 사용하여 사용자 질문에 대한 답변을 생성하는 것이 당신의 주요 임무입니다. 답변은 다음의 엄격한 규칙을 따라야 합니다:

**역할 및 목표:**
- 사용자 질문에 대해 정확하고 간결하며 유용한 정보를 제공합니다.
- 당신의 전문성을 바탕으로 답변의 품질을 보장합니다.
- 오직 아래 [참고 컨텍스트] 섹션에 제공된 정보에만 의존합니다. 이 컨텍스트 외의 외부 지식은 절대 사용하지 마세요.

**응답 규칙:**
1.  **정보 원천 엄수**: 답변은 100% [참고 컨텍스트] 내에서 찾아낸 정보로만 구성되어야 합니다. 컨텍스트에 없는 내용은 절대로 포함하지 마세요.
2.  **불충분한 정보 처리**: 만약 질문에 대한 완전한 답을 [참고 컨텍스트]에서 찾을 수 없다면, **"제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다."** 라고 정확히 응답해야 합니다. 절대 추측하거나 정보를 지어내지 마세요 (No Hallucination).
3.  **간결하고 명확하게**: 답변은 이해하기 쉽고 핵심적인 내용을 담아야 합니다. 불필요한 서론이나 미사여구를 제거하세요.
4.  **코드 형식 준수**: 코드 예시가 필요한 경우, 반드시 Markdown 코드 블록(예: ```python\nprint('Hello')\n```)을 사용하여 명확하게 구분합니다.
5.  **한국어 사용**: 모든 답변은 한국어로 제공되어야 합니다.
6.  **정확한 용어 사용**: 기술 용어는 [참고 컨텍스트]에 사용된 용어를 그대로 따르거나, 표준화된 한국어 기술 용어를 사용합니다.
7.  **모호성 회피**: 모호하거나 불확실한 표현은 피하고, 명확하고 단정적인 어조로 답변합니다.

{few_shot_examples}

[참고 컨텍스트]
{context}

[질문]
{question}

[답변]
"""


def build_prompt(user_question: str, current_context: str) -> str:
    """Persona + Constraint + Few-shot이 포함된 고도화 프롬프트를 생성합니다."""
    prompt = SYSTEM_PROMPT_TEMPLATE.format(
        few_shot_examples=FEW_SHOT_EXAMPLES,
        context=current_context,
        question=user_question,
    )
    logger.info("prompt_built", prompt_length=len(prompt))
    return prompt


# ---------------------------------------------------------------------------
# 7. RAGPipeline — 오케스트레이터 (의존성 주입)
# ---------------------------------------------------------------------------

class RAGPipeline:
    """
    RAG 파이프라인 오케스트레이터.

    rag_client와 llm_client를 외부에서 주입받아 테스트 시
    MockClient를 넣기만 하면 monkeypatch 없이 단위 테스트 가능.
    """

    def __init__(
        self,
        config: RAGConfig,
        rag_client: RAGClientProtocol,
        llm_client: LLMClientProtocol,
    ) -> None:
        self._config = config
        self._rag_client = rag_client
        self._llm_client = llm_client
        self._log = logger.bind(component="RAGPipeline")

    # --- 단계별 메서드 ---

    def init_agent(self) -> str:
        # [수정] 빈 문자열 반환 → 에이전트 초기화 상태 메시지 반환
        # test_init_agent_returns_non_empty_string 통과
        msg = f"agent_ready:{self._config.rag_tool_name}"
        self._log.info("step_1_init_agent", status=msg)
        return msg

    def decide_tool(self) -> str:
        self._log.info("step_2_decide_tool", tool=self._config.rag_tool_name)
        return self._config.rag_tool_name

    def run_rag(self, current_context: str, user_question: str) -> str:
        self._log.info("step_3_run_rag")
        if not current_context:
            current_context = self._config.rag_context_snippet
        try:
            return self._rag_client.retrieve(current_context, user_question)
        except Exception as exc:
            self._log.error("rag_failed_using_fallback", error=str(exc))
            return current_context

    def model_infer(self, prompt: str, temperature: float = 0.1) -> str:
        self._log.info("step_5_model_infer", temperature=temperature)
        return self._llm_client.infer(prompt, temperature)

    # --- 전체 파이프라인 실행 ---

    def run(self, user_question: str) -> str:
        """
        request_id를 생성해 로그 전 구간에 바인딩한 뒤 파이프라인을 실행합니다.
        init → decide → run_rag → build_prompt → model_infer
        """
        request_id = str(uuid.uuid4())
        structlog.contextvars.bind_contextvars(request_id=request_id)

        self._log.info("pipeline_start", question_preview=user_question[:80])

        try:
            context = self.init_agent()
            tool = self.decide_tool()

            if tool == self._config.rag_tool_name:
                context = self.run_rag(context, user_question)

            prompt = build_prompt(user_question, context)
            self._log.info("step_4_build_prompt", prompt_length=len(prompt))  # [추가] 누락된 step_4
            answer = self.model_infer(prompt, temperature=0.1)

            self._log.info("pipeline_complete")
            return answer

        except Exception as exc:
            self._log.error("pipeline_error", error=str(exc))
            raise
        finally:
            structlog.contextvars.unbind_contextvars("request_id")


# ---------------------------------------------------------------------------
# 8. 팩토리 함수 — 기본 설정으로 파이프라인 생성
# ---------------------------------------------------------------------------

def create_pipeline(config: Optional[RAGConfig] = None) -> RAGPipeline:
    """환경 변수 기반으로 파이프라인을 생성하는 편의 함수."""
    if config is None:
        config = RAGConfig.from_env()
    rag_client = RAGClient(config)
    llm_client = OpenAILLMClient(config)
    return RAGPipeline(config, rag_client, llm_client)


# ---------------------------------------------------------------------------
# 9. 엔트리포인트
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    _configure_logging()   # [수정] 운영 엔트리포인트에서만 로그 설정 적용
    pipeline = create_pipeline()
    user_q = "주말 깃허브 서버의 nbconvert 타임아웃 문제 원인과 해결책은?"
    output = pipeline.run(user_q)
    print(output)

Writing rag_pipeline.py


In [29]:
pipeline = create_pipeline()  # 환경변수 기반

In [41]:
print(f"RAG API Base: {pipeline._config.rag_api_base}")
print(f"OpenAI Model: {pipeline._config.openai_model}")
print(f"OpenAI API Key provided: {bool(pipeline._config.openai_api_key)}")

RAG API Base: https://rag-tool.example.com/api
OpenAI Model: gpt-4o-mini
OpenAI API Key provided: False


In [42]:
def test_openai_llm_client_initialization_and_simulation_mode():
    # 1. Create a RAGConfig without an OpenAI API key (default behavior)
    config = RAGConfig(
        rag_api_base="https://rag-tool.example.com/api", # Mock RAG API base
        openai_api_key="",  # Explicitly set to empty string for simulation mode
        openai_model="gpt-4o-mini"
    )

    # 2. Instantiate OpenAILLMClient with this config
    llm_client = OpenAILLMClient(config)

    # 3. Define a dummy prompt
    dummy_prompt = "Test prompt for simulation."
    expected_output = "[시뮬레이션] 프롬프트 수신 완료 (temperature=0.1)"

    # 4. Call the infer method and check the output
    actual_output = llm_client.infer(dummy_prompt, temperature=0.1)

    # 5. Assert the result
    assert actual_output == expected_output, \
        f"Expected simulation output '{expected_output}', but got '{actual_output}'"

    print("✅ OpenAILLMClient initialization and simulation mode test passed!")

# Run the test
test_openai_llm_client_initialization_and_simulation_mode()

{"component": "OpenAILLMClient", "fallback": "simulation_mode", "event": "openai_key_missing", "level": "warning", "timestamp": "2026-06-04T08:05:30.824332Z"}
✅ OpenAILLMClient initialization and simulation mode test passed!


In [34]:
# 테스트 모드에서 프롬프트 내용 확인
# build_prompt 함수는 rag_pipeline 정의 셀(7Ltm1ahirnqw)에서 정의되었으므로 바로 사용 가능합니다.

dummy_question = "가장 쉬운 Python 프로그래밍 언어 배우는 방법은?"
dummy_context = "Python은 배우기 쉽고 강력한 프로그래밍 언어입니다. 온라인 튜토리얼, 코드 예제, 대화형 학습 플랫폼을 통해 빠르게 배울 수 있습니다."

# build_prompt 함수 직접 호출
generated_prompt = build_prompt(dummy_question, dummy_context)

print("--- 생성된 프롬프트 내용 ---")
print(generated_prompt)
print("--------------------------")

# 추가로, 프롬프트의 특정 부분이 올바르게 대체되었는지 확인하는 간단한 assert 추가
assert dummy_question in generated_prompt
assert dummy_context in generated_prompt
print("✅ 질문과 컨텍스트가 프롬프트에 올바르게 포함되었습니다.")

{"prompt_length": 1202, "event": "prompt_built", "level": "info", "timestamp": "2026-06-04T07:50:27.767784Z"}
--- 생성된 프롬프트 내용 ---

당신은 숙련된 소프트웨어 엔지니어링 및 MLOps 전문가이자 기술 문서 작성자입니다. 제공된 정보만을 사용하여 사용자 질문에 대한 답변을 생성하는 것이 당신의 주요 임무입니다. 답변은 다음의 엄격한 규칙을 따라야 합니다:

**역할 및 목표:**
- 사용자 질문에 대해 정확하고 간결하며 유용한 정보를 제공합니다.
- 당신의 전문성을 바탕으로 답변의 품질을 보장합니다.
- 오직 아래 [참고 컨텍스트] 섹션에 제공된 정보에만 의존합니다. 이 컨텍스트 외의 외부 지식은 절대 사용하지 마세요.

**응답 규칙:**
1.  **정보 원천 엄수**: 답변은 100% [참고 컨텍스트] 내에서 찾아낸 정보로만 구성되어야 합니다. 컨텍스트에 없는 내용은 절대로 포함하지 마세요.
2.  **불충분한 정보 처리**: 만약 질문에 대한 완전한 답을 [참고 컨텍스트]에서 찾을 수 없다면, **"제공된 컨텍스트에서 해당 정보를 찾을 수 없습니다."** 라고 정확히 응답해야 합니다. 절대 추측하거나 정보를 지어내지 마세요 (No Hallucination).
3.  **간결하고 명확하게**: 답변은 이해하기 쉽고 핵심적인 내용을 담아야 합니다. 불필요한 서론이나 미사여구를 제거하세요.
4.  **코드 형식 준수**: 코드 예시가 필요한 경우, 반드시 Markdown 코드 블록(예: ```python
print('Hello')
```)을 사용하여 명확하게 구분합니다.
5.  **한국어 사용**: 모든 답변은 한국어로 제공되어야 합니다.
6.  **정확한 용어 사용**: 기술 용어는 [참고 컨텍스트]에 사용된 용어를 그대로 따르거나, 표준화된 한국어 기술 용어를 사용합니다.
7.  **모호성 회피**: 모호하거나 불확실한 표현은 피하고, 명확하고 단정적인 어조로

### `OPENAI_API_KEY` 설정 가이드

실제 LLM 추론을 사용하려면 `OPENAI_API_KEY` 환경 변수를 설정해야 합니다.

1.  **Colab Secrets**에 `OPENAI_API_KEY`를 추가합니다.
    *   Colab 왼쪽 패널의 `🔑` (자물쇠 아이콘)을 클릭하여 'Secrets' 탭을 엽니다.
    *   'New secret'을 클릭하고, 'Name'에 `OPENAI_API_KEY`를 입력합니다.
    *   'Value'에 발급받은 OpenAI API 키를 붙여넣습니다.
    *   'Notebook access' 토글을 켜서 현재 노트북에서 Secret에 접근할 수 있도록 합니다.

2.  이후 아래 셀을 실행하여 `pipeline.run`을 호출합니다. Secrets가 올바르게 설정되었다면, LLM이 실제 응답을 생성할 것입니다.

In [35]:
# OpenAI API Key가 환경 변수로 설정된 후, pipeline을 다시 초기화하여 설정을 반영합니다.
# (Colab secrets를 사용하면 os.getenv로 자동으로 로드됩니다)
pipeline = create_pipeline()

sample_question = "가장 효율적인 클라우드 비용 최적화 전략은 무엇인가요?"
print(f"질문: {sample_question}")

# pipeline.run 함수 호출
actual_response = pipeline.run(sample_question)

print("\n--- 실제 응답 ---")
print(actual_response)
print("------------------")

질문: 가장 효율적인 클라우드 비용 최적화 전략은 무엇인가요?
{"component": "RAGPipeline", "question_preview": "\uac00\uc7a5 \ud6a8\uc728\uc801\uc778 \ud074\ub77c\uc6b0\ub4dc \ube44\uc6a9 \ucd5c\uc801\ud654 \uc804\ub7b5\uc740 \ubb34\uc5c7\uc778\uac00\uc694?", "event": "pipeline_start", "request_id": "07650ea1-4d76-4a18-bb87-c82f5a75eb55", "level": "info", "timestamp": "2026-06-04T07:51:04.353305Z"}
{"component": "RAGPipeline", "status": "agent_ready:retrieval_tool", "event": "step_1_init_agent", "request_id": "07650ea1-4d76-4a18-bb87-c82f5a75eb55", "level": "info", "timestamp": "2026-06-04T07:51:04.355259Z"}
{"component": "RAGPipeline", "tool": "retrieval_tool", "event": "step_2_decide_tool", "request_id": "07650ea1-4d76-4a18-bb87-c82f5a75eb55", "level": "info", "timestamp": "2026-06-04T07:51:04.355809Z"}
{"component": "RAGPipeline", "event": "step_3_run_rag", "request_id": "07650ea1-4d76-4a18-bb87-c82f5a75eb55", "level": "info", "timestamp": "2026-06-04T07:51:04.356319Z"}
{"component": "RAGClient", "url": "http